# 05 - Full-scale run on a GPU (Colab / Kaggle)

The committed results in this repository were produced on a 2-core CPU with no
GPU, which caps their scale. This notebook runs what that machine cannot:

- **Real data** - ISIC 2018 Task 1 at 256px (needs a manual download; the cell
  below explains how, and the notebook falls back to synthetic if absent).
- **The 31M-parameter U-Net accuracy baseline** - roughly 30 s per training step
  on the CPU, so a single run there exceeds a day.
- **Longer schedules** at full resolution, where the differences between
  methods are least likely to be schedule artefacts.

Runtime on a free T4: about 40-60 minutes for the four-way comparison at 192px,
longer with ISIC at 256px.

**Set the runtime to GPU first** (Runtime, Change runtime type, T4 GPU).

In [ ]:
# Colab setup. Skip if running locally with the environment already installed.
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/HabibaSajid321/evidential-semi-supervised-lesion-segmentation"

if IN_COLAB:
    if not os.path.isdir("evidential-semi-supervised-lesion-segmentation"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("evidential-semi-supervised-lesion-segmentation")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
                   check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyyaml", "scipy", "pandas", "matplotlib", "tqdm"], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Set Runtime -> Change runtime type -> T4 GPU.")

## Optional: ISIC 2018

ISIC requires accepting terms of use, so it cannot be fetched automatically.
Either upload the two ZIPs to `data/isic2018/` and run `--extract`, or mount
Drive if you keep them there. Without them this notebook uses the synthetic
generator, which is a legitimate benchmark but not a claim about real data.

In [ ]:
from pathlib import Path
import subprocess, sys

isic_root = Path("data/isic2018")
isic_root.mkdir(parents=True, exist_ok=True)

# Uncomment to mount Drive and copy archives you have already downloaded:
# from google.colab import drive; drive.mount("/content/drive")
# !cp /content/drive/MyDrive/isic2018/*.zip data/isic2018/

if list(isic_root.glob("*.zip")):
    subprocess.run([sys.executable, "scripts/download_isic.py", "--extract"], check=True)

has_isic = (isic_root / "images").is_dir() and any((isic_root / "images").iterdir())
DATASET = "isic2018" if has_isic else "synthetic"
print("dataset:", DATASET)
if not has_isic:
    subprocess.run([sys.executable, "scripts/download_isic.py"], check=False)

## Full-scale four-way comparison

In [ ]:
from evissl.pipelines import run_comparison

if DATASET == "isic2018":
    SCALE = [
        "run.device=cuda", "data.name=isic2018", "data.image_size=256",
        "data.labeled_fraction=0.10", "data.batch_size=8", "data.mu=2",
        "data.num_workers=2", "model.width=32", "model.depth=4",
        "optim.epochs=60", "optim.steps_per_epoch=120", "optim.lr=0.002",
        "loss.kl_anneal_epochs=20", "semi.rampup_epochs=15",
    ]
else:
    SCALE = [
        "run.device=cuda", "data.image_size=192", "data.train_size=2000",
        "data.val_size=250", "data.test_size=500", "data.labeled_fraction=0.10",
        "data.batch_size=16", "data.mu=2", "data.num_workers=2",
        "model.width=32", "model.depth=4",
        "optim.epochs=60", "optim.steps_per_epoch=100",
        "loss.kl_anneal_epochs=20", "semi.rampup_epochs=15",
    ]

METHODS = [
    "configs/supervised_baseline.yaml",
    "configs/mean_teacher.yaml",
    "configs/fixmatch.yaml",
    "configs/evidential.yaml",
]

comparison = run_comparison(METHODS, overrides=SCALE, baseline="supervised_baseline",
                            keep_predictions=True)
display(comparison["table"].round(4))

In [ ]:
for metric, comparisons in comparison["comparisons"].items():
    print(f"--- {metric} ---")
    for c in comparisons:
        print("   ", c.summary())
    print()

## The 31M-parameter U-Net accuracy baseline

This is the run the CPU cannot do. It answers the question the efficiency table
alone cannot: does the 0.96M-parameter network actually give up accuracy for its
32x size reduction?

In [ ]:
from evissl.config import load_config
from evissl.pipelines import run_training

unet_scale = [o for o in SCALE if not o.startswith(("model.width", "model.depth", "optim.lr"))]
unet_cfg = load_config("configs/unet_baseline.yaml",
                       unet_scale + ["optim.lr=0.001", "run.name=unet_supervised"])
unet = run_training(unet_cfg)["result"]

# And the same architecture with the proposed training ideology.
unet_evidential_cfg = load_config(
    "configs/unet_baseline.yaml",
    unet_scale + ["optim.lr=0.001", "loss.supervised=evidential",
                  "semi.method=evidential", "run.name=unet_evidential"],
)
unet_evidential = run_training(unet_evidential_cfg)["result"]

rows = [r.summary_row() for r in
        list(comparison["results"].values()) + [unet, unet_evidential]]
frame = pd.DataFrame(rows)
display(frame[[c for c in ("run", "dice", "iou", "hd95", "boundary_f1", "ece",
                           "ause", "params_m", "gmacs", "latency_ms")
               if c in frame.columns]].round(4))

In [ ]:
from evissl.viz import plot_efficiency

fig = plot_efficiency(rows, x="params_m", y="dice", size="gmacs",
                      title="Accuracy against model size (full scale)")
plt.show()

## Figures and artefacts

In [ ]:
from evissl.report import write_comparison_figures, write_markdown_report

written = write_comparison_figures(comparison, "results")
print(write_markdown_report(comparison, "results/RESULTS_gpu.md"))
for path in written:
    print("  ", path)

In [ ]:
# Zip the artefacts so a Colab session's results survive it.
import shutil
shutil.make_archive("evissl_results", "zip", "results")
print("wrote evissl_results.zip")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("evissl_results.zip")

## Ablations at full scale

Optional and expensive - the sweep re-trains every method at four label budgets.
Budget roughly four times the comparison above.

In [ ]:
RUN_ABLATIONS = False

if RUN_ABLATIONS:
    from evissl.pipelines import run_component_ablation, run_labeled_fraction_sweep
    from evissl.report import write_sweep_figure

    display(run_component_ablation("configs/evidential.yaml", overrides=SCALE).round(4))
    sweep = run_labeled_fraction_sweep(
        ["configs/supervised_baseline.yaml", "configs/fixmatch.yaml",
         "configs/evidential.yaml"],
        fractions=(0.05, 0.10, 0.20, 0.50),
        overrides=SCALE,
    )
    display(sweep["table"].round(4))
    print(write_sweep_figure(sweep, "results"))